In [ ]:
#CSV and Excel File - Structrured Data Processing

In [1]:
import pandas as pd
import os

In [ ]:
os.makedirs("data/structured_files", exist_ok=True) # “If the folder already exists, don’t crash. - create it”

In [3]:
data = {
        'Product': ['Laptop', 'Mouse', 'Keyboard', 'Monitor', 'Webcam'],
        'Category': ['Electronics', 'Accessories', 'Accessories', 'Electronics', 'Electronics'],
        'Price': [999.99, 29.99, 79.99, 299.99, 89.99],
        'Stock': [50, 200, 150, 75, 100],
        'Description': [
        'High-performance laptop with 16GB RAM and 512GB SSD',
        'Wireless optical mouse with ergonomic design',
        'Mechanical keyboard with RGB backlighting',
        '27-inch 4K monitor with HDR support',
        '1080p webcam with noise cancellation'
    ]
}
df = pd.DataFrame(data) # DataFrame is a table-like data structure in Pandas.
df.to_csv("data/structured_files/products.csv", index=False)
print(f'''
DataFrames allows:
filtering
sorting
grouping
analytics
CSV/Excel/database export
preprocessing for ML/LLMs
exist_ok=True

exists_ok = “Don’t crash if folder already exists.”     
index = false means we don’t want to include the row numbers in the CSV file like 0,1,2,3,4. We just want the data without those numbers.
''')


DataFrames allows:
filtering
sorting
grouping
analytics
CSV/Excel/database export
preprocessing for ML/LLMs
exist_ok=True

exists_ok = “Don’t crash if folder already exists.”     
index = false means we don’t want to include the row numbers in the CSV file like 0,1,2,3,4. We just want the data without those numbers.



In [5]:
# Save mas Excel with mutiple sheets
with pd.ExcelWriter("data/structured_files/inventory.xlsx") as writer:
    df.to_excel(writer, sheet_name="Products", index=False)

    # Add another sheet with summary statistics
    summary_data ={
        'Category': ['Electronics', 'Accessories'],
        'Total_Items': [125, 350],
        'Average_Price': [463.32, 54.99]
    }
    pd.DataFrame(summary_data).to_excel(writer, sheet_name="Summary", index=False)

In [6]:
## CSV Processing

In [8]:
from langchain_community.document_loaders import CSVLoader, UnstructuredCSVLoader 


In [12]:
#Method 1 : CSV Loader - Each row is a document
print("Method 1 : CSV Loader - Each row is a document")
csv_loader = CSVLoader(
    file_path="data/structured_files/products.csv",
    encoding="utf-8",
    csv_args={
        "delimiter": ",",
        "quotechar": '"',
        }
)
csv_docs = csv_loader.load()
print(f"Loaded {len(csv_docs)} documents using CSVLoader.")
print("First document content:")
print(f"Content: {csv_docs[1].page_content}")
print(f"Metadata: {csv_docs[1].metadata}")

Method 1 : CSV Loader - Each row is a document
Loaded 5 documents using CSVLoader.
First document content:
Content: Product: Mouse
Category: Accessories
Price: 29.99
Stock: 200
Description: Wireless optical mouse with ergonomic design
Metadata: {'source': 'data/structured_files/products.csv', 'row': 1}


In [ ]:
#Method 2 : Custom CSV Loader with pandas for better control
from typing import List
from langchain_core.documents import Document
print("Method 2 : Custom CSV Loader with pandas for better control")

def process_csv_intelligently(filepath: str) -> List[Document]:
    """Process CSV Intelligently and return list of Documents with structured content."""

    # Read CSV with pandas for better control
    df= pd.read_csv(filepath)

    # Store documents in a list
    documents = []

    #Strategy 1: One document per row with structured content
    for idx, row in df.iterrows():
        #Create structured content
        content =f"""Product Information:
        Name:{row['Product']}
        Category:{row['Category']}
        Price:${row['Price']}
        Stock:{row['Stock']} units
        Description:{row['Description']}"""
    
    #Create document with rich metadata
        document = Document(
            page_content = content,
            metadata={
                "source": filepath,
                "row_index": idx,
                "product_name": row['Product'],
                "category": row['Category'],
                "price": row['Price'],
                "stock": row['Stock'],
                "data_type": "product_info"
            }
        )
        documents.append(document)
    return documents

Method 2 : Custom CSV Loader for better control


In [17]:
process_csv_intelligently("data/structured_files/products.csv")

[Document(metadata={'source': 'data/structured_files/products.csv', 'row_index': 0, 'product_name': 'Laptop', 'category': 'Electronics', 'price': 999.99, 'stock': 50, 'data_type': 'product_info'}, page_content='Product Information:\n        Name:Laptop\n        Category:Electronics\n        Price:$999.99\n        Stock:50 units\n        Description:High-performance laptop with 16GB RAM and 512GB SSD'),
 Document(metadata={'source': 'data/structured_files/products.csv', 'row_index': 1, 'product_name': 'Mouse', 'category': 'Accessories', 'price': 29.99, 'stock': 200, 'data_type': 'product_info'}, page_content='Product Information:\n        Name:Mouse\n        Category:Accessories\n        Price:$29.99\n        Stock:200 units\n        Description:Wireless optical mouse with ergonomic design'),
 Document(metadata={'source': 'data/structured_files/products.csv', 'row_index': 2, 'product_name': 'Keyboard', 'category': 'Accessories', 'price': 79.99, 'stock': 150, 'data_type': 'product_info'}

In [ ]:
### Excel Processing

In [19]:
##Method 1: Using Pandas for Full Control
print("Method 1: Pandas Based Excel Processing for Full Control")
def process_excel_with_pandas(filepath: str) -> List[Document]:
    """Pandas based Excel Procesing"""
    documents= []

    # Read al sheets
    excel_file= pd.ExcelFile(filepath)

    for sheet_name in excel_file.sheet_names:
        # read each sheet into a DataFrame
        df = pd.read_excel(excel_file, sheet_name=sheet_name)

        # create a document for each row with sheet name in metadata
        sheet_content=f"Sheet: {sheet_name}\n\n"
        sheet_content += f"Columns: {', '.join(df.columns)}\n\n"
        sheet_content += f"Rows: {len(df)}\n\n"
        sheet_content += df.to_string(index=False) # Convert DataFrame to string without row numbers

        document = Document(
            page_content= sheet_content,
            metadata={
                "source": filepath,
                "sheet_name": sheet_name,
                "num_rows": len(df),
                "num_columns": len(df.columns),
                "data_type": "excel_sheet"
            }
        )
        documents.append(document)
    return documents

Method 1: Pandas Based Excel Processing for Full Control


In [22]:
excel_docs = process_excel_with_pandas("data/structured_files/inventory.xlsx")
print(f"Processed {len(excel_docs)} sheets")
excel_docs

Processed 2 sheets


[Document(metadata={'source': 'data/structured_files/inventory.xlsx', 'sheet_name': 'Products', 'num_rows': 5, 'num_columns': 5, 'data_type': 'excel_sheet'}, page_content='Sheet: Products\n\nColumns: Product, Category, Price, Stock, Description\n\nRows: 5\n\n Product    Category  Price  Stock                                         Description\n  Laptop Electronics 999.99     50 High-performance laptop with 16GB RAM and 512GB SSD\n   Mouse Accessories  29.99    200        Wireless optical mouse with ergonomic design\nKeyboard Accessories  79.99    150           Mechanical keyboard with RGB backlighting\n Monitor Electronics 299.99     75                 27-inch 4K monitor with HDR support\n  Webcam Electronics  89.99    100                1080p webcam with noise cancellation'),
 Document(metadata={'source': 'data/structured_files/inventory.xlsx', 'sheet_name': 'Summary', 'num_rows': 2, 'num_columns': 3, 'data_type': 'excel_sheet'}, page_content='Sheet: Summary\n\nColumns: Category, Tot

In [25]:
# Method 2: Using UnstructuredExcelLoader for Simplicity
from langchain_community.document_loaders import UnstructuredExcelLoader
print("Method 2: UnstructuredExcelLoader for Simplicity")
try:
    excel_loader = UnstructuredExcelLoader(
        file_path="data/structured_files/inventory.xlsx",
        mode="elements"
    )
    excel_docs = excel_loader.load()
    print(f"Processed {len(excel_docs)} elements from Excel file.")
    print(f"First document content: {excel_docs[0].page_content}")
    print(f"First document metadata: {excel_docs[0].metadata}")
except Exception as e:
    print(f"Error occurred while processing Excel file: {e}")

Method 2: UnstructuredExcelLoader for Simplicity
Processed 2 elements from Excel file.
First document content: Product Category Price Stock Description Laptop Electronics 999.99 50 High-performance laptop with 16GB RAM and 512GB SSD Mouse Accessories 29.99 200 Wireless optical mouse with ergonomic design Keyboard Accessories 79.99 150 Mechanical keyboard with RGB backlighting Monitor Electronics 299.99 75 27-inch 4K monitor with HDR support Webcam Electronics 89.99 100 1080p webcam with noise cancellation
First document metadata: {'source': 'data/structured_files/inventory.xlsx', 'file_directory': 'data/structured_files', 'filename': 'inventory.xlsx', 'last_modified': '2026-05-17T18:36:54', 'page_name': 'Products', 'page_number': 1, 'text_as_html': '<table><tr><td>Product</td><td>Category</td><td>Price</td><td>Stock</td><td>Description</td></tr><tr><td>Laptop</td><td>Electronics</td><td>999.99</td><td>50</td><td>High-performance laptop with 16GB RAM and 512GB SSD</td></tr><tr><td>Mouse